# Celeb-DF-v2 face extraction (run once, on Kaggle)

Produces the 224x224 face crops every training run needs. Celeb-DF-v2 is already
mounted here, so nothing is downloaded.

**Before running:** add `reubensuju/celeb-df-v2` via *+ Add Input*, and set
*Settings > Accelerator* to **None** (this is CPU-bound) and *Persistence* to
**Files only** so a reclaimed session keeps its progress.

Runtime is roughly 1-2 hours for all 6,229 videos. Re-running skips whatever is
already done, so a disconnect costs nothing.

## 1. Get the code

In [ ]:
# Option A -- clone from GitHub (edit the URL):
REPO = ""  # e.g. "https://github.com/<you>/dip-hfc-deepfake.git"

import os, subprocess, sys
from pathlib import Path

CODE = None
if REPO:
    if not os.path.exists("/kaggle/working/code"):
        subprocess.run(["git", "clone", REPO, "/kaggle/working/code"], check=True)
    CODE = "/kaggle/working/code"
else:
    # Option B -- find whichever mounted input actually holds the package,
    # instead of assuming what Kaggle slugified the dataset title into.
    for hit in Path("/kaggle/input").rglob("models/hfc.py"):
        CODE = str(hit.parents[2])
        break

if CODE is None:
    print("Code not found. Inputs currently mounted:")
    for d in sorted(Path("/kaggle/input").iterdir()):
        print("   ", d.name)
    raise SystemExit(
        "Upload dip-hfc-code.zip as a Kaggle dataset, attach it via + Add Input, re-run."
    )

sys.path.insert(0, CODE)
os.chdir(CODE)
print("code at", CODE)
print(os.listdir(CODE))

In [ ]:
!pip install -q mediapipe albumentations timm 2>&1 | tail -2
import mediapipe, cv2
print("mediapipe", mediapipe.__version__, "| opencv", cv2.__version__)

## 2. Locate the dataset

Kaggle mirrors vary in how deeply they nest the three folders, so find them rather than assume.

In [ ]:
from pathlib import Path

ROOT = None
for p in Path("/kaggle/input").rglob("Celeb-synthesis"):
    if p.is_dir():
        ROOT = p.parent
        break
assert ROOT, "Celeb-synthesis not found -- did you add the dataset as an input?"

print("dataset root:", ROOT)
for sub in ("Celeb-real", "Celeb-synthesis", "YouTube-real"):
    d = ROOT / sub
    print(f"  {sub:18s} {len(list(d.glob('*.mp4'))) if d.is_dir() else 'MISSING':>6}")
print("\nExpect 590 / 5639 / 300 (Table I of the paper).")

## 3. Smoke test on 20 videos first

Three minutes here beats discovering a bad detector two hours in.

In [ ]:
!python -m src.data.extract_faces \
    --root {ROOT} --out /kaggle/working/_smoke \
    --frames 32 --size 224 --workers 2 --limit 20

In [ ]:
# Eyeball a few crops -- are they tight, upright, actually faces?
import matplotlib.pyplot as plt, cv2
from pathlib import Path

jpgs = sorted(Path("/kaggle/working/_smoke").rglob("*.jpg"))
print(len(jpgs), "crops")
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, p in zip(axes.ravel(), jpgs[:: max(1, len(jpgs) // 12)]):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(p.parent.name[:18], fontsize=7)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Full extraction

The long one. Safe to re-run after a disconnect.

In [ ]:
!python -m src.data.extract_faces \
    --root {ROOT} --out /kaggle/working/faces \
    --frames 32 --size 224 --workers 4

## 5. Build the split manifests

In [ ]:
from src.data.splits import build_splits, summarise

m = build_splits(ROOT, "/kaggle/working/faces/splits.json")
print(summarise(m))

## 6. Pack for distribution

~200k loose files is painful for Kaggle to save and slow for Colab to fetch.
A single tar copies to Colab's local SSD in a couple of minutes.

In [ ]:
!cd /kaggle/working && tar -cf faces.tar faces && ls -lh faces.tar
!find /kaggle/working/faces -name '*.jpg' | wc -l

Now *Save Version > Save & Run All*, then publish the output as a **private
Kaggle dataset**. Every Colab account pulls it with:

```bash
kaggle datasets download -d <you>/celebdf-hfc-faces -p /content --unzip
tar -xf /content/faces.tar -C /content
```